# Padding and Stride
:label:`sec_padding`

- Recall the convolution example in :numref:`fig_correlation`:
  - Input size: $3 \times 3$
  - Kernel size: $2 \times 2$
  - Output size: $2 \times 2$

- In general, for input shape $n_\textrm{h} \times n_\textrm{w}$ and kernel shape $k_\textrm{h} \times k_\textrm{w}$:
  - The output shape is:
    $$
    (n_\textrm{h} - k_\textrm{h} + 1) \times (n_\textrm{w} - k_\textrm{w} + 1)
    $$
  - This is because the kernel can only **slide** so far before **running out of pixels** to cover.

- In the following, we explore techniques that provide **more control** over output size:
  - **Padding**
  - **Strided convolutions**

- **Motivation**:
  - Since kernels usually have **height and width > 1**,
    - Repeated convolutions lead to **shrinking output dimensions**.

- Example:
  - Starting with a $240 \times 240$ image,
  - Ten layers of $5 \times 5$ convolutions reduce it to $200 \times 200$,
  - Losing **30%** of the image,
  - And **discarding important boundary information**.

- **Padding**:
  - The most common method to address this issue.
  - Adds pixels around the input’s border to **preserve spatial dimensions**.

- **Strided convolutions**:
  - Used when we **intentionally want to reduce dimensionality**.
  - Helpful when the **original resolution is too large** to process directly.


In [1]:
import torch
from torch import nn

## Padding

- One challenge with convolutional layers is the **loss of pixels near the image borders**:
  - As shown in :numref:`img_conv_reuse`, **pixel utilization** varies based on:
    - The **kernel size**.
    - The **position** within the image.
  - **Corner pixels** are often **rarely used**.

![Pixel utilization for convolutions of size $1 \times 1$, $2 \times 2$, and $3 \times 3$ respectively.](../img/conv-reuse.svg)
:label:`img_conv_reuse`

- Although small kernels only lose a few pixels per layer,
  - This **accumulates** across **many layers**.
  - Eventually, **valuable information at the edges** may be lost.

- A straightforward solution:
  - **Add extra pixels** (called *padding*) around the input’s boundary.
  - This **increases the effective input size**.

- Typically:
  - Padding values are set to **zero** (zero-padding).

- In :numref:`img_conv_pad`:
  - A $3 \times 3$ input is padded to become $5 \times 5$.
  - The resulting output becomes a $4 \times 4$ matrix.
  - The shaded regions illustrate the computation of the **first output element**:
    $$
    0\times0 + 0\times1 + 0\times2 + 0\times3 = 0
    $$

![Two-dimensional cross-correlation with padding.](../img/conv-pad.svg)
:label:`img_conv_pad`


- In general, if we add:
  - $p_\textrm{h}$ rows of padding (roughly half on top and half on bottom), and
  - $p_\textrm{w}$ columns of padding (roughly half on the left and half on the right),
  - Then the **output shape** becomes:

  $$
  (n_\textrm{h} - k_\textrm{h} + p_\textrm{h} + 1) \times (n_\textrm{w} - k_\textrm{w} + p_\textrm{w} + 1).
  $$

- This means the **height and width** of the output will increase by:
  - $p_\textrm{h}$ (height), and
  - $p_\textrm{w}$ (width).

- In many cases, we choose:
  - $p_\textrm{h} = k_\textrm{h} - 1$
  - $p_\textrm{w} = k_\textrm{w} - 1$
  - This makes the **input and output** have the **same height and width**.

- This setup makes it easier to **predict output shapes** during network construction.

- **Padding strategy**:
  - If $k_\textrm{h}$ is **odd**, pad $p_\textrm{h}/2$ rows on **both sides** (top and bottom).
  - If $k_\textrm{h}$ is **even**, one approach is to pad:
    - $\lceil p_\textrm{h}/2 \rceil$ rows on **top**, and
    - $\lfloor p_\textrm{h}/2 \rfloor$ rows on **bottom**.
  - Apply the same strategy for width padding.


- CNNs commonly use **convolution kernels with odd dimensions**, such as 1, 3, 5, or 7.

- **Benefits of odd-sized kernels**:
  - They allow us to **preserve dimensionality** by:
    - Padding with the **same number of rows** on top and bottom.
    - Padding with the **same number of columns** on left and right.

- This practice provides a **clerical advantage**:
  - For a 2D tensor `X`, if:
    - The kernel size is **odd**, and
    - Padding is **symmetric** on all sides,
  - Then the output will have the **same height and width** as the input.
  - In this case, the output `Y[i, j]` is computed by:
    - **Cross-correlating** the input with the kernel,
    - With the kernel window **centered at `X[i, j]`**.

- Example:
  - We create a 2D convolutional layer with:
    - Kernel size: **3 × 3**
    - Padding: **1 pixel on all sides**
  - Input size: **8 × 8**
  - Output size: also **8 × 8**


In [2]:
# We define a helper function to calculate convolutions. It initializes the
# convolutional layer weights and performs corresponding dimensionality
# elevations and reductions on the input and output
def comp_conv2d(conv2d, X):
    # (1, 1) indicates that batch size and the number of channels are both 1
    X = X.reshape((1, 1) + X.shape)
    Y = conv2d(X)
    # Strip the first two dimensions: examples and channels
    return Y.reshape(Y.shape[2:])

# 1 row and column is padded on either side, so a total of 2 rows or columns
# are added
conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1)
X = torch.rand(size=(8, 8))
comp_conv2d(conv2d, X).shape

torch.Size([8, 8])

- When the **height and width** of the convolution kernel are **different**:
  - We can still make the **output and input** have the **same dimensions** by:

- **Setting different padding values** for:
  - **Height** and
  - **Width**

- This ensures that the output maintains the same **spatial size** as the input.


In [3]:
# We use a convolution kernel with height 5 and width 3. The padding on either
# side of the height and width are 2 and 1, respectively
conv2d = nn.LazyConv2d(1, kernel_size=(5, 3), padding=(2, 1))
comp_conv2d(conv2d, X).shape

torch.Size([8, 8])

## Stride

- In **cross-correlation**, we typically slide the **convolution window**:
  - From the **upper-left corner** of the input,
  - **Down** and **to the right**, covering all positions.

- So far, we have used a **stride of 1**:
  - Moving the window **one element at a time**.

- However, we can also **skip positions** by moving the window **more than one element** at a time.
  - This is useful for:
    - **Computational efficiency**,
    - **Downsampling** the input,
    - Or when using a **large kernel** to capture a broad image area.

- The number of rows and columns the window moves per step is called the **stride**.

- Example in :numref:`img_conv_stride`:
  - **Stride = 3** vertically and **2** horizontally.
  - The shaded regions show:
    - The **output elements**, and
    - The **input and kernel elements** used in the computation.

  - Example calculations:
    $$
    0\times0 + 0\times1 + 1\times2 + 2\times3 = 8 \\
    0\times0 + 6\times1 + 0\times2 + 0\times3 = 6
    $$

- Behavior:
  - When generating the **second element of the first column**,
    - The window moves **down three rows**.
  - When generating the **second element of the first row**,
    - The window moves **two columns to the right**.
  - If the window slides beyond the input boundary,
    - **No output is produced** unless **additional padding** is added.

![Cross-correlation with strides of 3 and 2 for height and width, respectively.](../img/conv-stride.svg)
:label:`img_conv_stride`


- In general, if the **stride** for height is $s_\textrm{h}$ and for width is $s_\textrm{w}$, then the **output shape** is:

  $$
  \lfloor(n_\textrm{h} - k_\textrm{h} + p_\textrm{h} + s_\textrm{h}) / s_\textrm{h} \rfloor \times 
  \lfloor(n_\textrm{w} - k_\textrm{w} + p_\textrm{w} + s_\textrm{w}) / s_\textrm{w} \rfloor.
  $$

- If we set:
  - $p_\textrm{h} = k_\textrm{h} - 1$
  - $p_\textrm{w} = k_\textrm{w} - 1$

  - Then the formula simplifies to:

  $$
  \lfloor(n_\textrm{h} + s_\textrm{h} - 1)/s_\textrm{h}\rfloor \times 
  \lfloor(n_\textrm{w} + s_\textrm{w} - 1)/s_\textrm{w}\rfloor.
  $$

- Moreover, if the input height and width are **divisible** by the strides, then:

  $$
  \text{Output shape} = (n_\textrm{h} / s_\textrm{h}) \times (n_\textrm{w} / s_\textrm{w})
  $$

- In the following example, we **set both strides to 2**:
  - This means the input's height and width will each be **halved**.


In [4]:
conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1, stride=2)
comp_conv2d(conv2d, X).shape

torch.Size([4, 4])

- Let's look at (**a slightly more complicated example**).


In [5]:
conv2d = nn.LazyConv2d(1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))
comp_conv2d(conv2d, X).shape

torch.Size([2, 2])

## Summary and Discussion

- **Padding** increases the **height and width** of the output.
  - Often used to **preserve the input's spatial dimensions**,
  - Prevents **undesirable shrinkage** across layers,
  - Ensures that **all pixels are used equally** during computation.

- Typically, we choose **symmetric padding**:
  - Padding $(p_\textrm{h}, p_\textrm{w})$ means:
    - Add $p_\textrm{h}$ rows of padding to top and bottom,
    - Add $p_\textrm{w}$ columns to left and right.
  - When $p_\textrm{h} = p_\textrm{w}$, we just write **padding $p$**.

- The same applies to **stride**:
  - When $s_\textrm{h} = s_\textrm{w}$, we write **stride $s$**.
  - A stride **reduces resolution**:
    - For stride $n$, output is roughly **$1/n$ the size** of input.
  - **Default values**:
    - **Padding = 0**, **stride = 1**.

- All padding so far has used **zero-padding**:
  - Easy to implement,
  - Computationally efficient—**no need for extra memory allocation**,
  - Enables CNNs to **learn positional patterns** (e.g., recognizing where padding is).

- There are other padding methods:
  - **Nonzero padding**, such as **mirror** or **replicated** padding,
  - :citet:`Alsallakh.Kokhlikyan.Miglani.ea.2020` reviewed many such methods,
  - However, **zero-padding remains standard** unless artifacts suggest alternatives.
